# SMART Baseline Evaluation (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/wosac-sim-agents-experiments/blob/main/experiments/smart-baseline/notebooks/smart_baseline_eval_colab.ipynb)

This notebook owns the **baseline checkpoint evaluation stage**:
- bind the persisted processed validation data,
- run rollout export / closed-loop simulation,
- compute official metrics,
- write a strict-contract-ready baseline evaluation artifact.


In [1]:
# Step 1: Sync this repo and bootstrap deterministic Colab runtime
import os
import subprocess
import sys

REPO_URL = "https://github.com/achyutmorang/wosac-sim-agents-experiments.git"
REPO_DIR = "/content/wosac-sim-agents-experiments"

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", "main", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
for p in (REPO_DIR, os.path.join(REPO_DIR, "src")):
    if p not in sys.path:
        sys.path.insert(0, p)

from src.platform import bootstrap_colab_runtime_with_config, wosac_colab_runtime_config
runtime_cfg = wosac_colab_runtime_config(repo_url=REPO_URL, repo_branch="main", repo_dir=REPO_DIR)
bootstrap = bootstrap_colab_runtime_with_config(runtime_cfg)
print("repo_rev:", bootstrap.repo_sync.repo_rev)

if bootstrap.setup.result.get("restart_required", False):
    raise RuntimeError("Runtime restart required. Restart runtime and rerun this cell.")


[repo] existing checkout: /content/wosac-sim-agents-experiments
[drive] /content/drive already mounted
[drive] Verified read/write access: /content/drive/MyDrive/wosac_experiments
[setup] Starting deterministic environment bootstrap
[setup] $ /usr/bin/python3 -c import jax, waymax, numpy as np, pandas as pd, scipy, sklearn; from numpy._core.umath import _center, _expandtabs; print('ok', np.__version__, pd.__version__, scipy.__version__, sklearn.__version__, jax.__version__)
[setup] Core runtime already healthy; skipping dependency install (ok 2.2.6 2.2.3 1.14.1 1.6.1 0.7.2).
[setup] $ /usr/bin/python3 -c import numpy as np; from numpy._core.umath import _center, _expandtabs; print(np.__version__, np.__file__)
[setup] NumPy probe passed (2.2.6 /usr/local/lib/python3.12/dist-packages/numpy/__init__.py).
[setup] $ /usr/bin/python3 -c import jax, waymax, numpy as np, pandas as pd, scipy, sklearn; from numpy._core.umath import _center, _expandtabs; print('ok', np.__version__, pd.__version__

In [2]:
# Suggested defaults: baseline-only rollout + official metrics on persisted validation data.
%env WOSAC_RUN_MODE=full
%env WOSAC_AUTO_RESUME=1
%env SMART_PROCESSED_DATA_ROOT=/content/drive/MyDrive/wosac_experiments/datasets/waymo_processed
%env SMART_VAL_CONFIG=experiments/smart-baseline/configs/validation_scalable_paper_repro.yaml
%env WOSAC_SCENARIO_TFRECORDS=gs://waymo_open_dataset_motion_v_1_3_1/uncompressed/scenario/validation/validation.tfrecord-*
%env WOSAC_RECOMPUTE_METRICS=1


env: WOSAC_RUN_MODE=full
env: WOSAC_AUTO_RESUME=1
env: SMART_PROCESSED_DATA_ROOT=/content/drive/MyDrive/wosac_experiments/datasets/waymo_processed
env: SMART_VAL_CONFIG=experiments/smart-baseline/configs/validation_scalable_paper_repro.yaml
env: WOSAC_SCENARIO_TFRECORDS=gs://waymo_open_dataset_motion_v_1_3_1/uncompressed/scenario/validation/validation.tfrecord-*
env: WOSAC_RECOMPUTE_METRICS=1


In [3]:
# Step 2: Load config, resolve checkpoint/output roots, and bind persisted validation data
import hashlib
import importlib
import json
import os
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

from src.workflows import bootstrap_experiment_pack, load_experiment_config


def _bool_env(name: str, default: bool) -> bool:
    text = os.environ.get(name, '').strip().lower()
    if not text:
        return bool(default)
    return text in {'1', 'true', 'yes', 'y', 'on'}


def _latest_dir(root: Path, pattern: str) -> Path | None:
    if not root.exists():
        return None
    paths = [p for p in root.glob(pattern) if p.exists() and p.is_dir()]
    if not paths:
        return None
    paths.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return paths[0]


def _latest_ckpt(search_root: Path) -> str:
    if (not search_root.exists()) or (not search_root.is_dir()):
        return ''
    ckpts = [p for p in search_root.rglob('*.ckpt') if p.is_file()]
    if not ckpts:
        return ''
    ckpts.sort(key=lambda p: (p.stat().st_mtime, str(p)), reverse=True)
    return str(ckpts[0])


def _count_processed(split_dir: Path) -> int:
    files = [p for p in split_dir.glob('*.pkl') if p.is_file()]
    files += [p for p in split_dir.glob('*.pickle') if p.is_file()]
    return len(files)


def _bind_canonical_processed_root(actual_root: Path, canonical_root: Path) -> dict:
    actual_root = actual_root.expanduser()
    canonical_root = canonical_root.expanduser()
    if actual_root.resolve() == canonical_root.resolve():
        canonical_root.mkdir(parents=True, exist_ok=True)
        return {'mode': 'direct', 'canonical_root': str(canonical_root), 'actual_root': str(actual_root)}
    canonical_root.parent.mkdir(parents=True, exist_ok=True)
    if canonical_root.is_symlink():
        if canonical_root.resolve() != actual_root.resolve():
            canonical_root.unlink()
            canonical_root.symlink_to(actual_root, target_is_directory=True)
        return {'mode': 'symlink', 'canonical_root': str(canonical_root), 'actual_root': str(actual_root)}
    if canonical_root.exists():
        has_entries = any(canonical_root.iterdir())
        if has_entries and canonical_root.resolve() != actual_root.resolve():
            raise RuntimeError(f'Cannot bind {canonical_root} to {actual_root}: canonical path already exists and is non-empty.')
        if not has_entries:
            canonical_root.rmdir()
            canonical_root.symlink_to(actual_root, target_is_directory=True)
            return {'mode': 'symlink', 'canonical_root': str(canonical_root), 'actual_root': str(actual_root)}
    canonical_root.symlink_to(actual_root, target_is_directory=True)
    return {'mode': 'symlink', 'canonical_root': str(canonical_root), 'actual_root': str(actual_root)}


def _ensure_colab_gcs_auth_if_needed(paths: list[str]) -> None:
    if 'google.colab' not in sys.modules:
        return
    if not any(str(p).strip().startswith('gs://') for p in paths if str(p).strip()):
        return
    from google.colab import auth  # type: ignore
    auth.authenticate_user()
    print('Colab auth: OK')
    try:
        active = subprocess.check_output(
            ['bash', '-lc', "gcloud auth list --filter=status:ACTIVE --format='value(account)'"],
            text=True,
        ).strip()
        print('Active gcloud account:', active or '<none>')
    except Exception as exc:
        print(f'Active gcloud account lookup failed: {exc}')


def _ensure_waymo_open_dataset() -> None:
    try:
        importlib.import_module('waymo_open_dataset')
        print('waymo_open_dataset already installed.')
        return
    except Exception:
        pass

    py_major, py_minor = sys.version_info[:2]
    if (py_major, py_minor) >= (3, 12):
        print('Python 3.12+ detected. Trying compatibility install (no pinned deps) first ...')
        cmd = [sys.executable, '-m', 'pip', 'install', '--upgrade', '--no-deps', 'waymo-open-dataset-tf-2-12-0==1.6.7']
        print('Running:', ' '.join(cmd))
        subprocess.run(cmd, check=True)
        importlib.import_module('waymo_open_dataset')
        print('Compatibility install succeeded on Python 3.12+.')
        return

    cmd = [sys.executable, '-m', 'pip', 'install', '--upgrade', 'waymo-open-dataset-tf-2-12-0==1.6.7']
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True)
    importlib.import_module('waymo_open_dataset')
    print('Waymo package install succeeded.')


EXPERIMENT_SLUG = 'smart-baseline'
bundle = bootstrap_experiment_pack(slug=EXPERIMENT_SLUG, repo_root='.')
cfg = load_experiment_config(slug=EXPERIMENT_SLUG, repo_root='.')
display(bundle.summary_table)

RUN = dict(cfg.get('run', {}))
SMART = dict(cfg.get('smart', {}))
SMART_PROFILES = dict(SMART.get('profiles', {}))

RUN_NAME = str(RUN.get('run_name', 'dev'))
RUN_PREFIX = str(RUN.get('run_prefix', 'smart_baseline'))
PERSIST_ROOT = str(RUN.get('persist_root', '/content/drive/MyDrive/wosac_experiments'))
cfg_hash = hashlib.sha256(json.dumps(cfg, sort_keys=True).encode('utf-8')).hexdigest()
persist_run_dir = Path(PERSIST_ROOT) / f'{RUN_PREFIX}_{RUN_NAME}'
persist_run_dir.mkdir(parents=True, exist_ok=True)
outputs_root = persist_run_dir / 'outputs'
outputs_root.mkdir(parents=True, exist_ok=True)

RUN_MODE = os.environ.get('WOSAC_RUN_MODE', 'full').strip().lower() or 'full'
if RUN_MODE not in {'pilot', 'full'}:
    RUN_MODE = 'full'

AUTO_RESUME = _bool_env('WOSAC_AUTO_RESUME', True)
explicit_run_tag = os.environ.get('WOSAC_RUN_TAG', '').strip()
if explicit_run_tag:
    RUN_TAG = explicit_run_tag
elif AUTO_RESUME:
    latest = _latest_dir(outputs_root, '*')
    RUN_TAG = latest.name if latest is not None else datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
else:
    RUN_TAG = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUN_OUTPUT_DIR = outputs_root / RUN_TAG
RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SIM_MANIFESTS_DIR = RUN_OUTPUT_DIR / 'simulation_manifests'
ROLLOUT_PROTO_DIR = RUN_OUTPUT_DIR / 'rollout_protos'
OFFICIAL_METRICS_DIR = RUN_OUTPUT_DIR / 'official_metrics'
for d in [SIM_MANIFESTS_DIR, ROLLOUT_PROTO_DIR, OFFICIAL_METRICS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SMART_PROFILE = os.environ.get('SMART_BASELINE_PROFILE', str(SMART.get('profile', 'paper_repro'))).strip() or 'paper_repro'
SMART_EFFECTIVE = dict(SMART)
if SMART_PROFILE in SMART_PROFILES:
    SMART_EFFECTIVE.update(dict(SMART_PROFILES[SMART_PROFILE]))

processed_root_override = os.environ.get('SMART_PROCESSED_DATA_ROOT', '').strip()
if processed_root_override:
    SMART_EFFECTIVE['processed_data_root'] = processed_root_override
SMART_VAL_CONFIG = os.environ.get('SMART_VAL_CONFIG', str(SMART_EFFECTIVE.get('val_config', 'experiments/smart-baseline/configs/validation_scalable_paper_repro.yaml'))).strip() or str(SMART_EFFECTIVE.get('val_config', 'experiments/smart-baseline/configs/validation_scalable_paper_repro.yaml'))
BASELINE_CKPT_ROOT = Path(os.environ.get('SMART_BASELINE_CKPT_ROOT', str(persist_run_dir / 'checkpoints'))).expanduser()
BASELINE_CKPT = os.environ.get('SMART_BASELINE_CKPT', '').strip() or _latest_ckpt(BASELINE_CKPT_ROOT)
assert BASELINE_CKPT, 'Could not auto-discover baseline checkpoint. Set SMART_BASELINE_CKPT.'
SMOKE_EVAL_DEFAULT = 'smart_baseline_smoke' in BASELINE_CKPT
DEFAULT_MAX_EVAL_SCENARIOS = 32 if SMOKE_EVAL_DEFAULT else 0
OFFICIAL_REQUIRED_ROLLOUT_COUNT = 32
DEFAULT_EVAL_ROLLOUT_COUNT = OFFICIAL_REQUIRED_ROLLOUT_COUNT if SMOKE_EVAL_DEFAULT else int(RUN.get('n_rollouts_per_scenario', OFFICIAL_REQUIRED_ROLLOUT_COUNT))
MAX_EVAL_SCENARIOS = int((os.environ.get('SMART_MAX_EVAL_SCENARIOS', str(DEFAULT_MAX_EVAL_SCENARIOS)).strip() or str(DEFAULT_MAX_EVAL_SCENARIOS)))
EVAL_ROLLOUT_COUNT = int((os.environ.get('SMART_ROLLOUT_COUNT', str(DEFAULT_EVAL_ROLLOUT_COUNT)).strip() or str(DEFAULT_EVAL_ROLLOUT_COUNT)))
DEFAULT_PROGRESS_EVERY = 4 if (MAX_EVAL_SCENARIOS > 0 and MAX_EVAL_SCENARIOS <= 64) else 25
EXPORT_PROGRESS_EVERY = int((os.environ.get('SMART_EXPORT_PROGRESS_EVERY', str(DEFAULT_PROGRESS_EVERY)).strip() or str(DEFAULT_PROGRESS_EVERY)))
EXPORT_FLUSH_EVERY = int((os.environ.get('SMART_EXPORT_FLUSH_EVERY', str(EXPORT_PROGRESS_EVERY)).strip() or str(EXPORT_PROGRESS_EVERY)))

PROCESSED_ROOT = Path(str(SMART_EFFECTIVE.get('processed_data_root', '/content/SMART/data/waymo_processed'))).expanduser()
CANONICAL_PROCESSED_ROOT = Path('/content/SMART/data/waymo_processed')
PATH_BINDING = _bind_canonical_processed_root(PROCESSED_ROOT, CANONICAL_PROCESSED_ROOT)
PROCESSED_VAL_DIR = CANONICAL_PROCESSED_ROOT / 'validation'
processed_val_count = _count_processed(PROCESSED_VAL_DIR) if PROCESSED_VAL_DIR.exists() else 0
if processed_val_count <= 0:
    raise RuntimeError('Processed validation data is missing. Run smart_data_prep_colab.ipynb first.')

SCENARIO_PROTO_PATH = os.environ.get('WOSAC_SCENARIO_PROTO_PATH', '').strip()
SCENARIO_PROTO_DIR = os.environ.get('WOSAC_SCENARIO_PROTO_DIR', '').strip()
SCENARIO_TFRECORDS = os.environ.get('WOSAC_SCENARIO_TFRECORDS', '').strip()
SCENARIO_SET_ID = os.environ.get('WOSAC_SCENARIO_SET_ID', 'womd_validation').strip() or 'womd_validation'
EVALUATOR_ID = os.environ.get('WOSAC_EVALUATOR_ID', 'waymo_open_dataset.sim_agents_metrics.challenge_2025').strip()
SIM_SEED = int(os.environ.get('WOSAC_SIM_SEED', '2'))
STRICT_ROLLOUT_VALIDATION = _bool_env('WOSAC_STRICT_ROLLOUT_VALIDATION', False)
RECOMPUTE_METRICS = _bool_env('WOSAC_RECOMPUTE_METRICS', True)

if any([SCENARIO_PROTO_PATH, SCENARIO_PROTO_DIR, SCENARIO_TFRECORDS]) and EVAL_ROLLOUT_COUNT != OFFICIAL_REQUIRED_ROLLOUT_COUNT:
    raise RuntimeError(
        f'Official Sim Agents evaluation requires SMART_ROLLOUT_COUNT={OFFICIAL_REQUIRED_ROLLOUT_COUNT}; '
        f'got {EVAL_ROLLOUT_COUNT}. Bound SMART_MAX_EVAL_SCENARIOS instead.'
    )

_ensure_colab_gcs_auth_if_needed([SCENARIO_PROTO_PATH, SCENARIO_PROTO_DIR, SCENARIO_TFRECORDS])
_ensure_waymo_open_dataset()

print('RUN_TAG:', RUN_TAG)
print('RUN_OUTPUT_DIR:', RUN_OUTPUT_DIR)
print('SIM_MANIFESTS_DIR:', SIM_MANIFESTS_DIR)
print('ROLLOUT_PROTO_DIR:', ROLLOUT_PROTO_DIR)
print('OFFICIAL_METRICS_DIR:', OFFICIAL_METRICS_DIR)
print('SMART_PROFILE:', SMART_PROFILE)
print('SMART_VAL_CONFIG:', SMART_VAL_CONFIG)
print('BASELINE_CKPT:', BASELINE_CKPT)
print('PROCESSED_VAL_COUNT:', processed_val_count)
print('SMOKE_EVAL_DEFAULT:', SMOKE_EVAL_DEFAULT)
print('MAX_EVAL_SCENARIOS:', MAX_EVAL_SCENARIOS)
print('EVAL_ROLLOUT_COUNT:', EVAL_ROLLOUT_COUNT)
print('EXPORT_PROGRESS_EVERY:', EXPORT_PROGRESS_EVERY)
print('EXPORT_FLUSH_EVERY:', EXPORT_FLUSH_EVERY)
print('path_binding:', json.dumps(PATH_BINDING, indent=2, sort_keys=True))
print('SCENARIO_PROTO_PATH:', SCENARIO_PROTO_PATH or '<optional>')
print('SCENARIO_PROTO_DIR:', SCENARIO_PROTO_DIR or '<optional>')
print('SCENARIO_TFRECORDS:', SCENARIO_TFRECORDS or '<required for official metrics if no proto path/dir>')
print('STRICT_ROLLOUT_VALIDATION:', STRICT_ROLLOUT_VALIDATION)
print('RECOMPUTE_METRICS:', RECOMPUTE_METRICS)
print('cfg_hash:', cfg_hash)


,field,value
0,slug,smart-baseline
1,title,SMART Baseline Wrapper
2,objective,Reproduce SMART baseline with thin wrapper flo...
3,notebook,experiments/smart-baseline/notebooks/smart-bas...
4,workflow,src/workflows/smart_baseline_flow.py
5,config_file,/content/wosac-sim-agents-experiments/configs/...


Colab auth: OK
Active gcloud account: morang.achyut@gmail.com
Python 3.12+ detected. Trying compatibility install (no pinned deps) first ...
Running: /usr/bin/python3 -m pip install --upgrade --no-deps waymo-open-dataset-tf-2-12-0==1.6.7
Compatibility install succeeded on Python 3.12+.
RUN_TAG: 20260404T162859Z
RUN_OUTPUT_DIR: /content/drive/MyDrive/wosac_experiments/smart_baseline_dev/outputs/20260404T162859Z
SIM_MANIFESTS_DIR: /content/drive/MyDrive/wosac_experiments/smart_baseline_dev/outputs/20260404T162859Z/simulation_manifests
ROLLOUT_PROTO_DIR: /content/drive/MyDrive/wosac_experiments/smart_baseline_dev/outputs/20260404T162859Z/rollout_protos
OFFICIAL_METRICS_DIR: /content/drive/MyDrive/wosac_experiments/smart_baseline_dev/outputs/20260404T162859Z/official_metrics
SMART_PROFILE: paper_repro
SMART_VAL_CONFIG: experiments/smart-baseline/configs/validation_scalable_paper_repro.yaml
BASELINE_CKPT: /content/drive/MyDrive/wosac_experiments/smart_baseline_dev/checkpoints/smart_baseline

In [4]:
# Step 3: Build baseline rollout command plan
from src.workflows import run_smart_eval_flow

models = [
    {
        'model_id': 'smart_baseline',
        'checkpoint_path': BASELINE_CKPT,
        'scenario_rollouts_path': str(ROLLOUT_PROTO_DIR / 'smart_baseline.binproto'),
        'env': {},
    }
]

flow_bundle = run_smart_eval_flow(
    run_tag=RUN_TAG,
    run_name=RUN_NAME,
    run_prefix=RUN_PREFIX,
    persist_root=PERSIST_ROOT,
    repo_root='.',
    smart_repo_url=str(SMART_EFFECTIVE.get('repo_url', 'https://github.com/rainmaker22/SMART.git')),
    smart_repo_branch=str(SMART_EFFECTIVE.get('branch', 'main')),
    smart_repo_dir=str(SMART_EFFECTIVE.get('repo_dir', '/content/SMART')),
    smart_val_config=str(SMART_VAL_CONFIG),
    sync_smart_repo=True,
    models=models,
    strict_contract=False,
    require_metrics_binding=False,
    verify_checkpoint_hash=False,
    n_rollouts=int(EVAL_ROLLOUT_COUNT),
    seed=int(SIM_SEED),
    max_scenarios=int(MAX_EVAL_SCENARIOS),
    progress_every=int(EXPORT_PROGRESS_EVERY),
    flush_every=int(EXPORT_FLUSH_EVERY),
    scenario_proto_path=SCENARIO_PROTO_PATH,
    scenario_proto_dir=SCENARIO_PROTO_DIR,
    scenario_tfrecords=SCENARIO_TFRECORDS,
    strict_rollout_validation=STRICT_ROLLOUT_VALIDATION,
)

print('summary:')
print(json.dumps(flow_bundle.summary, indent=2, sort_keys=True))
print('baseline rollout command:')
print(flow_bundle.models[0]['rollout_cmd'])
print('baseline progress json:')
print(flow_bundle.models[0].get('progress_json_path', ''))


summary:
{
  "binding_keys": [
    "manifest_sha256",
    "model_id",
    "scenario_set_hash",
    "evaluator_id",
    "metrics_config_hash",
    "n_rollouts",
    "num_history_seconds",
    "num_future_seconds",
    "seed"
  ],
  "compatibility_keys": [
    "scenario_set_hash",
    "evaluator_id",
    "metrics_config_hash",
    "n_rollouts",
    "num_history_seconds",
    "num_future_seconds"
  ],
  "config_hash": "ea9f47c98366a0d05b7995e32b53a11db5524271fbe03e4c04823799493307b9",
  "created_utc": "2026-04-04T19:43:51Z",
  "flush_every": 4,
  "manifests_dir": "",
  "max_scenarios": 32,
  "metrics_dir": "",
  "n_rollouts": 32,
  "num_contract_invalid": 0,
  "num_models": 1,
  "persist_root": "/content/drive/MyDrive/wosac_experiments",
  "progress_every": 4,
  "repo_root": "/content/wosac-sim-agents-experiments",
  "require_metrics_binding": false,
  "run_name": "dev",
  "run_prefix": "smart_baseline",
  "run_tag": "20260404T162859Z",
  "scenario_proto_dir": "",
  "scenario_proto_path":

In [5]:
# Step 4: Optional rollout execution (baseline only)
import subprocess

RUN_SETUP = _bool_env('SMART_RUN_SETUP', True)
RUN_ROLLOUT = _bool_env('SMART_RUN_ROLLOUT', True)
model = flow_bundle.models[0]
rollout_path = Path(str(model.get('scenario_rollouts_path', '')).strip()).expanduser()
progress_json_path = Path(str(model.get('progress_json_path', '')).strip()).expanduser()
rollout_exists = rollout_path.exists() and rollout_path.stat().st_size > 0

print('RUN_SETUP:', RUN_SETUP)
print('RUN_ROLLOUT:', RUN_ROLLOUT)
print('rollout_exists:', rollout_exists)
print('rollout_path:', rollout_path)
print('progress_json_path:', progress_json_path)


def _run_cmd(cmd: str, idx: int, total: int, *, label: str) -> None:
    print(f'[{label} {idx}/{total}] {cmd}')
    merged_cmd = f'export PYTHONUNBUFFERED=1; {cmd}'
    process = subprocess.Popen(
        ['bash', '-lc', merged_cmd],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    recent_lines = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
        recent_lines.append(line)
        if len(recent_lines) > 80:
            recent_lines.pop(0)
    return_code = int(process.wait())
    if return_code == 0:
        return
    diag = {
        'failed_command': cmd,
        'exit_code': return_code,
        'rollout_path': str(rollout_path),
        'progress_json_path': str(progress_json_path),
        'recent_output': ''.join(recent_lines[-20:]),
        'smart_val_config': str(flow_bundle.summary.get('smart_val_config', '')),
    }
    print('step4_diagnostics:')
    print(json.dumps(diag, indent=2, sort_keys=True))
    raise subprocess.CalledProcessError(return_code, ['bash', '-lc', merged_cmd])

if RUN_SETUP:
    setup_cmd = (
        f'cd {REPO_DIR} && '
        f'python {REPO_DIR}/scripts/ensure_smart_train_runtime.py '
        f'--smart-repo-dir {flow_bundle.summary.get("smart_repo_dir", "/content/SMART")}'
    )
    _run_cmd(setup_cmd, 1, 1 + int(bool(RUN_ROLLOUT)), label='setup')

if RUN_ROLLOUT:
    _run_cmd(model['rollout_cmd'], 1 + int(bool(RUN_SETUP)), 1 + int(bool(RUN_SETUP)), label='simulate:smart_baseline')
else:
    print('SMART_RUN_ROLLOUT=0 -> skipping rollout execution.')

print('Rollout execution block done.')


RUN_SETUP: True
RUN_ROLLOUT: True
rollout_exists: True
rollout_path: /content/drive/MyDrive/wosac_experiments/smart_baseline_dev/outputs/20260404T162859Z/rollout_protos/smart_baseline.binproto
progress_json_path: /content/drive/MyDrive/wosac_experiments/smart_baseline_dev/outputs/20260404T162859Z/rollout_protos/smart_baseline.progress.json
[setup 1/2] cd /content/wosac-sim-agents-experiments && python /content/wosac-sim-agents-experiments/scripts/ensure_smart_train_runtime.py --smart-repo-dir /content/SMART
[smart-train-setup] missing pytorch_lightning: ModuleNotFoundError: No module named 'pytorch_lightning'
[smart-train-setup] $ /usr/bin/python3 -m pip install --upgrade pytorch-lightning>=2.4,<2.6
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 831.6/831.6 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 22.1 MB/s eta 0:00:00
[smart-train-setup] using PyG wheel index: https://data.pyg.org/whl/torch-2.10.0+cu128.html
[smart-train-setup] $ /usr/bin/python3

In [7]:
# Step 5: Compute official metrics, write manifest, and run strict contract checks
import hashlib

from src.workflows import compute_official_metrics_from_rollouts, load_json_mapping, run_smart_eval_flow, write_json, write_simulation_manifest

metrics_json_path = OFFICIAL_METRICS_DIR / 'smart_baseline.json'
rollout_path = Path(str(flow_bundle.models[0].get('scenario_rollouts_path', '')).strip()).expanduser()
progress_json_path = Path(str(flow_bundle.models[0].get('progress_json_path', '')).strip()).expanduser()
progress_payload = {}
if progress_json_path.exists():
    progress_payload = load_json_mapping(progress_json_path)
    if str(progress_payload.get('status', '')).strip().lower() != 'complete':
        raise RuntimeError(
            f'Rollout export is not complete yet (status={progress_payload.get("status", "unknown")}). Finish Step 4 first.'
        )
if (not rollout_path.exists()) or rollout_path.stat().st_size == 0:
    raise RuntimeError('Missing rollout proto. Run Step 4 first.')
if (not SCENARIO_PROTO_PATH) and (not SCENARIO_PROTO_DIR) and (not SCENARIO_TFRECORDS):
    raise RuntimeError('Official metrics require scenario source data. Set WOSAC_SCENARIO_TFRECORDS or a scenario proto path/dir.')

result = compute_official_metrics_from_rollouts(
    scenario_rollouts_path=str(rollout_path),
    scenario_proto_path=SCENARIO_PROTO_PATH,
    scenario_proto_dir=SCENARIO_PROTO_DIR,
    scenario_tfrecords=SCENARIO_TFRECORDS,
    challenge_type='SIM_AGENTS',
    output_metrics_json=str(metrics_json_path),
)
scenario_ids = [str(s).strip() for s in (result.get('scenario_ids', []) or []) if str(s).strip()]
scenario_wire = json.dumps(sorted(scenario_ids), ensure_ascii=True, separators=(',', ':'))
scenario_set_hash = 'sha256:' + hashlib.sha256(scenario_wire.encode('utf-8')).hexdigest()
metrics_config_hash = 'sha256:' + str(result.get('metrics_config_sha256', '')).strip()
manifest_path = SIM_MANIFESTS_DIR / 'smart_baseline_simulation_manifest.json'
manifest = write_simulation_manifest(
    manifest_path,
    {
        'created_utc': datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
        'run_tag': RUN_TAG,
        'run_name': RUN_NAME,
        'run_prefix': RUN_PREFIX,
        'model_id': 'smart_baseline',
        'checkpoint_path': BASELINE_CKPT,
        'checkpoint_sha256': '',
        'validate_cmd': str(flow_bundle.models[0].get('validate_cmd', '')),
        'rollout_cmd': str(flow_bundle.models[0].get('rollout_cmd', '')),
        'scenario_rollouts_path': str(rollout_path),
        'scenario_proto_path': SCENARIO_PROTO_PATH,
        'scenario_proto_dir': SCENARIO_PROTO_DIR,
        'scenario_tfrecords': SCENARIO_TFRECORDS,
        'scenario_set_id': SCENARIO_SET_ID,
        'scenario_set_hash': scenario_set_hash,
        'evaluator_id': EVALUATOR_ID,
        'metrics_config_hash': metrics_config_hash,
        'n_rollouts': int(EVAL_ROLLOUT_COUNT),
        'num_history_seconds': int(RUN.get('num_history_seconds', 1)),
        'num_future_seconds': int(RUN.get('num_future_seconds', 8)),
        'seed': int(SIM_SEED),
        'smart_repo_rev': str((flow_bundle.summary.get('smart_repo_sync', {}) or {}).get('repo_rev', 'unknown')),
        'source_notebook': 'smart_baseline_eval_colab.ipynb',
    },
)
metrics_payload = load_json_mapping(metrics_json_path)
metrics_payload.update(
    {
        'manifest_sha256': manifest.get('manifest_sha256', ''),
        'model_id': 'smart_baseline',
        'scenario_set_hash': scenario_set_hash,
        'evaluator_id': EVALUATOR_ID,
        'metrics_config_hash': metrics_config_hash,
        'n_rollouts': int(EVAL_ROLLOUT_COUNT),
        'num_history_seconds': int(RUN.get('num_history_seconds', 1)),
        'num_future_seconds': int(RUN.get('num_future_seconds', 8)),
        'seed': int(SIM_SEED),
    }
)
write_json(metrics_json_path, metrics_payload)

strict_bundle = run_smart_eval_flow(
    run_tag=RUN_TAG,
    run_name=RUN_NAME,
    run_prefix=RUN_PREFIX,
    persist_root=PERSIST_ROOT,
    repo_root='.',
    smart_repo_url=str(SMART_EFFECTIVE.get('repo_url', 'https://github.com/rainmaker22/SMART.git')),
    smart_repo_branch=str(SMART_EFFECTIVE.get('branch', 'main')),
    smart_repo_dir=str(SMART_EFFECTIVE.get('repo_dir', '/content/SMART')),
    smart_val_config=str(SMART_VAL_CONFIG),
    sync_smart_repo=False,
    n_rollouts=int(EVAL_ROLLOUT_COUNT),
    seed=int(SIM_SEED),
    max_scenarios=int(MAX_EVAL_SCENARIOS),
    progress_every=int(EXPORT_PROGRESS_EVERY),
    flush_every=int(EXPORT_FLUSH_EVERY),
    models=[
        {
            'model_id': 'smart_baseline',
            'checkpoint_path': BASELINE_CKPT,
            'manifest_json': str(manifest_path),
            'metrics_json': str(metrics_json_path),
            'scenario_rollouts_path': str(rollout_path),
            'env': {},
        }
    ],
    manifests_dir=str(SIM_MANIFESTS_DIR),
    metrics_dir=str(OFFICIAL_METRICS_DIR),
    strict_contract=True,
    require_metrics_binding=True,
    verify_checkpoint_hash=True,
)

print('official_metrics_result:')
print(json.dumps({
    'metrics_json': str(metrics_json_path),
    'realism_meta_metric': (result.get('metrics', {}) or {}).get('realism_meta_metric'),
    'scenario_count': result.get('scenario_count'),
    'scenario_set_hash': scenario_set_hash,
    'metrics_config_hash': metrics_config_hash,
}, indent=2, sort_keys=True))
print('strict_summary:')
print(json.dumps(strict_bundle.summary, indent=2, sort_keys=True))


ImportError: cannot import name 'load_json_mapping' from 'src.workflows' (/content/wosac-sim-agents-experiments/src/workflows/__init__.py)

In [8]:
# Step 6: Contract gate table
import pandas as pd

rows = []
for model in strict_bundle.models:
    rows.append({
        'model_id': model.get('model_id'),
        'contract_valid': bool(model.get('contract_valid', False)),
        'contract_errors': ';'.join(model.get('contract_errors', [])),
        'metrics_source': model.get('metrics_source'),
        'realism_meta_metric': (model.get('metrics', {}) or {}).get('realism_meta_metric'),
    })

df = pd.DataFrame(rows)
display(df)
invalid = [r for r in rows if not bool(r['contract_valid'])]
if invalid:
    raise RuntimeError(f'Strict contract failed for {len(invalid)} model(s). Check contract_errors in table.')
print('All baseline eval contract checks passed.')


NameError: name 'strict_bundle' is not defined

In [9]:
# Step 7: Write baseline evaluation artifact (Drive)
payload = {
    'run_id': 'smart_baseline_eval_v0',
    'status': str(strict_bundle.summary.get('status', 'unknown')),
    'run_tag': RUN_TAG,
    'run_output_dir': str(RUN_OUTPUT_DIR),
    'rollout_proto_path': str(rollout_path),
    'simulation_manifest': str(manifest_path),
    'official_metrics_json': str(metrics_json_path),
    'path_binding': PATH_BINDING,
    'summary': strict_bundle.summary,
    'models': strict_bundle.models,
    'artifacts': strict_bundle.artifacts,
    'notes': [
        'Baseline checkpoint evaluation notebook: rollout export and official metrics in one place.',
    ],
    'provenance': {
        'config_hash': cfg_hash,
        'created_utc': datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
        'experiment_slug': EXPERIMENT_SLUG,
        'source_notebook': 'smart_baseline_eval_colab.ipynb',
    },
}

drive_path = RUN_OUTPUT_DIR / 'smart_baseline_eval_v0.json'
write_json(drive_path, payload)
print('drive_path:', drive_path)


NameError: name 'strict_bundle' is not defined